# Project 1
## Gender Prediction

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from names_dataset import NameDataset

### Create fake names

In [2]:
fake_gen = Faker()

In [3]:
def generate_name(prob: float = 0.5) -> str:
    """
    Generate and return random names. Male and female names are split based on `prob`

    Args:
        prob (float): Probability of female name generation
    """

    if np.random.rand() > prob:
        return fake_gen.name_female()
    return fake_gen.name_male()

### Create a `DataFrame` with fake names

In [4]:
df = pd.DataFrame({"Name": [generate_name() for _ in range(500)]})

In [5]:
df

,Name
0,Karen Noble
1,John Norris
2,Daniel Washington
3,Charles Anderson
4,Mary Matthews
...,...
495,Kara Kaiser
496,Molly Johnson
497,Angela Harmon
498,Molly Gonzalez


### Extract first name and last name

In [6]:
name_suffixes = {"MD", "PhD", "MBA", "DVM", "DDS", "III", "Jr.", "Sr."}

In [7]:
name_prefixes = {"Dr.", "Mr.", "Mrs.", "Ms."}

In [8]:
def extract_fname(full_name: str) -> str:
    """
    Extract the first name from the given full name.
    """
    
    split_txt = full_name.split()
    if len(split_txt) > 2:
        if split_txt[0] in name_prefixes:
            return split_txt[1]
    return split_txt[0]

In [9]:
def extract_lname(full_name: str) -> str:
    """
    Extract the first name from the given full name.
    """
    split_txt = full_name.split()
    if len(split_txt) == 4:
        return split_txt[2]
    if len(split_txt) == 3:
        if split_txt[-1] in name_suffixes:
            return split_txt[1]
    return split_txt[-1]

In [10]:
df["First Name"] = df["Name"].apply(extract_fname)

In [11]:
df["Last Name"] = df["Name"].apply(extract_lname)

In [12]:
df

,Name,First Name,Last Name
0,Karen Noble,Karen,Noble
1,John Norris,John,Norris
2,Daniel Washington,Daniel,Washington
3,Charles Anderson,Charles,Anderson
4,Mary Matthews,Mary,Matthews
...,...,...,...
495,Kara Kaiser,Kara,Kaiser
496,Molly Johnson,Molly,Johnson
497,Angela Harmon,Angela,Harmon
498,Molly Gonzalez,Molly,Gonzalez


### Predict gender

In [13]:
nd = NameDataset()

In [14]:
def name_to_gender(name: str) -> str | None:
    """
    Predict the gender of the given name using `NameDataset` from `names_dataset` library.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["first_name"]
    if query is None:
        return None 
    return max(query["gender"], key=query["gender"].get)

In [15]:
def gender_prob(name):
    """
    Return the probability of the given name being male or female.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["first_name"]
    if query is None:
        return None 
    return max(query["gender"].values())   

In [16]:
df["Gender"] = df["First Name"].apply(name_to_gender)

In [17]:
df["Gender Probability"] = df["First Name"].apply(gender_prob)

In [18]:
df

,Name,First Name,Last Name,Gender,Gender Probability
0,Karen Noble,Karen,Noble,Female,0.981
1,John Norris,John,Norris,Male,0.981
2,Daniel Washington,Daniel,Washington,Male,0.988
3,Charles Anderson,Charles,Anderson,Male,0.984
4,Mary Matthews,Mary,Matthews,Female,0.985
...,...,...,...,...,...
495,Kara Kaiser,Kara,Kaiser,Female,0.791
496,Molly Johnson,Molly,Johnson,Female,0.980
497,Angela Harmon,Angela,Harmon,Female,0.992
498,Molly Gonzalez,Molly,Gonzalez,Female,0.980


### Predict Country

In [19]:
def name_to_country(name: str) -> str | None:
    """
    Predict the country of origin based on the given name.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["last_name"]
    if query is None:
        return None 
    return max(query["country"], key=query["country"].get)

In [20]:
def country_prob(name: str) -> float | None:
    """
    Return the probability of the given name belonging to the predicted country.
    """

    if not isinstance(name, str):
        return None
    query = nd.search(name)["last_name"]
    if query is None:
        return None 
    return max(query["country"].values())

I used last name as the indicator of nationality because it is more stable and predictable.

In [21]:
df["Country"] = df["Last Name"].apply(name_to_country)

In [22]:
df["Country Probability"] = df["Last Name"].apply(country_prob)

In [23]:
df

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
0,Karen Noble,Karen,Noble,Female,0.981,United Kingdom,0.349
1,John Norris,John,Norris,Male,0.981,United States,0.500
2,Daniel Washington,Daniel,Washington,Male,0.988,United States,0.901
3,Charles Anderson,Charles,Anderson,Male,0.984,United States,0.599
4,Mary Matthews,Mary,Matthews,Female,0.985,United Kingdom,0.411
...,...,...,...,...,...,...,...
495,Kara Kaiser,Kara,Kaiser,Female,0.791,Germany,0.500
496,Molly Johnson,Molly,Johnson,Female,0.980,United States,0.646
497,Angela Harmon,Angela,Harmon,Female,0.992,United States,0.857
498,Molly Gonzalez,Molly,Gonzalez,Female,0.980,United States,0.275


In [24]:
df[["Country Probability", "Gender Probability"]] = df[["Country Probability", "Gender Probability"]].astype(np.float32)

In [25]:
df[df["Name"].str.split().apply(len) > 2]

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
26,Mrs. Tracy Brown,Tracy,Brown,Female,0.937,United States,0.515
27,Richard Stone Jr.,Richard,Stone,Male,0.992,United States,0.475
31,Mr. Thomas Clay,Thomas,Clay,Male,0.992,United States,0.589
43,Laura Conrad DDS,Laura,Conrad,Female,0.993,United States,0.437
64,Jennifer Mcdonald MD,Jennifer,Mcdonald,Female,0.993,United States,0.448
74,Cameron Harding DVM,Cameron,Harding,Male,0.934,United Kingdom,0.560
75,Kim Lewis MD,Kim,Lewis,Female,0.799,United States,0.518
115,Mrs. Yvette Burke,Yvette,Burke,Female,0.990,United States,0.397
185,Mr. Paul Baker,Paul,Baker,Male,0.990,United States,0.462
215,Lauren Ward MD,Lauren,Ward,Female,0.985,United Kingdom,0.450


Names with prefixes and/or suffixes are correctly processed

In [26]:
df["Country"].value_counts()

Country
United States     378
United Kingdom     65
Colombia           21
Mexico             11
Ireland             5
Germany             5
Brazil              4
Malaysia            3
France              2
Denmark             2
Nigeria             2
Saudi Arabia        1
Hong Kong           1
Name: count, dtype: int64

In [28]:
df[df["Country"] == "Germany"]

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
117,Eric Fischer,Eric,Fischer,Male,0.992,Germany,0.520
153,Leslie Mayer,Leslie,Mayer,Female,0.820,Germany,0.319
338,Joshua Haas,Joshua,Haas,Male,0.987,Germany,0.289
485,Jaime Klein,Jaime,Klein,Male,0.965,Germany,0.313
495,Kara Kaiser,Kara,Kaiser,Female,0.791,Germany,0.500


In [32]:
df[df["Country"].isin(["Denmark", "France"])]

,Name,First Name,Last Name,Gender,Gender Probability,Country,Country Probability
80,Sean Bernard,Sean,Bernard,Male,0.989,France,0.673
210,Steven Christensen,Steven,Christensen,Male,0.992,Denmark,0.533
236,Mario Hansen,Mario,Hansen,Male,0.990,Denmark,0.430
383,Robert Richard,Robert,Richard,Male,0.992,France,0.491


In [29]:
df["Gender"].value_counts()

Gender
Female    264
Male      236
Name: count, dtype: int64

In [33]:
df.to_csv("Data/Gender.csv")